# 优化方法

此前，你一直使用梯度下降更新参数并最小化代价。本 Notebook 将介绍几种更高级的优化方法，它们可以加快训练，也可能获得更低的最终代价。合适的优化算法有时能将取得良好结果所需的时间从数天缩短至几小时。

读完本 Notebook，你将能够： 

* 应用优化方法，例如（随机）梯度下降、Momentum、RMSProp 和 Adam
* 使用随机mini-batch来加速收敛并提高优化

梯度下降在代价函数 $J$ 上“下坡”。把它想象成试图做到这一点： 
<img src="images/cost.jpg" style="width:650px;height:300px;">
<div style="text-align:center;"><u> <b>图 1</b> </u>：<b>最小化代价就像寻找丘陵地形中的最低点</b><br>训练时，每一步都沿某个方向更新参数，尝试到达尽可能低的位置。</div>

**符号**：像往常一样，$\frac{\partial J}{\partial a } = $ `da` 对于任何变量 `a`。

让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/deep-neural-network/supplement/QWEnZ/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录
- [1- 依赖库](#1)
- [2 - 梯度下降](#2)
    - [练习 1 - update_parameters_with_gd](#ex-1)
- [3 - mini-batch 梯度下降](#3)
    - [练习 2 - random_mini_batches](#ex-2)
- [4 - Momentum（动量）](#4)
    - [练习 3 - initialize_velocity](#ex-3)
    - [练习 4 - update_parameters_with_momentum](#ex-4)
- [5 - Adam](#5)
    - [练习 5 - initialize_adam](#ex-5)
    - [练习 6 - update_parameters_with_adam](#ex-6)
- [6 - 使用不同优化算法的模型](#6)
    - [6.1 - mini-batch 梯度下降](#6-1)
    - [6.2 - 带动量的 mini-batch 梯度下降](#6-2)
    - [6.3 - 使用 Adam 的 mini-batch 优化](#6-3)
    - [6.4 - 总结](#6-4)
- [7 - 学习率衰减和调度](#7)
    - [7.1 - 每次迭代进行学习率衰减](#7-1)
        - [练习 7 - update_lr](#ex-7)
    - [7.2 - 固定间隔调度](#7-2)
        - [练习 8 - schedule_lr_decay](#ex-8)
    - [7.3 - 对每种优化方法使用学习率衰减](#7-3)
        - [7.3.1 - 梯度下降与学习率衰减](#7-3-1)
        - [7.3.2 - 带动量和学习率衰减的梯度下降](#7-3-2)
        - [7.3.3 - 带学习率衰减的 Adam](#7-3-3)
    - [7.4 - 用不同的方法实现相似的性能](#7-4)

<a name='1'></a>
## 1- 依赖库

In [ ]:
### v1.3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import math
import sklearn
import sklearn.datasets

from opt_utils_v1a import load_params_and_grads, initialize_parameters, forward_propagation, backward_propagation
from opt_utils_v1a import compute_cost, predict, predict_dec, plot_decision_boundary, load_dataset
from copy import deepcopy
from testCases import *
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0) # 设置绘图的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

<a name='2'></a>
## 2 - 梯度下降

梯度下降（GD）是一种简单的机器学习优化方法。如果每一步都使用全部 $m$ 个样本计算梯度并更新参数，就称为批量梯度下降（Batch Gradient Descent）。

<a name='ex-1'></a>
### 练习 1 - update_parameters_with_gd

实现梯度下降更新规则。对于 $l = 1, ..., L$，梯度下降规则是：

$$
W^{[l]} = W^{[l]} - \alpha \text{ } dW^{[l]} \tag{1}
$$

$$
b^{[l]} = b^{[l]} - \alpha \text{ } db^{[l]} \tag{2}
$$

，其中L是层数，$\alpha$是学习率。所有参数应存储在 `parameters` 字典中。请注意，迭代器 `l` 在 `for` 循环中从 1 开始，因为第一个参数是 $W^{[1]}$ 和 $b^{[1]}$。 

In [ ]:
# GRADED FUNCTION: update_parameters_with_gd

def update_parameters_with_gd(parameters, grads, learning_rate):
    """
    使用梯度下降一步更新参数
    
    参数：
    parameters -- 包含要更新的参数的Python字典：
                    parameters['W' + str(l)] = Wl
                    parameters['b' + str(l)] = bl
    grads -- 包含梯度的 python 字典，用于更新每个参数：
                    grads['dW' + str(l)] = dWl
                    grads['db' + str(l)] = dbl
    learning_rate -- 学习率，标量。
    
    返回：
    parameters -- 包含更新参数的Python字典 
    """
    L = len(parameters) // 2 # 神经网络的层数

    # 每个参数的更新规则
    for l in range(1, L + 1):
        # （约2行）
        # parameters["W" + str(l)] =  
        # parameters["b" + str(l)] = 
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    return parameters

In [ ]:
parameters, grads, learning_rate = update_parameters_with_gd_test_case()
learning_rate = 0.01
parameters = update_parameters_with_gd(parameters, grads, learning_rate)

print("W1 =\n" + str(parameters["W1"]))
print("b1 =\n" + str(parameters["b1"]))
print("W2 =\n" + str(parameters["W2"]))
print("b2 =\n" + str(parameters["b2"]))

update_parameters_with_gd_test(update_parameters_with_gd)

它的一个变体是随机梯度下降 (SGD)，它相当于 mini-batch 梯度下降，其中每个 mini-batch 只有 1 个示例。你刚刚实现的更新规则不会改变。变化在于，你一次只能在一个训练样本上计算梯度，而不是在整个训练集上计算。下面的代码示例说明了随机梯度下降和（批量）梯度下降之间的差异。 

- **（批次）梯度下降**：

``` python
X = data_input
Y = labels
m = X.shape[1]  # 训练样本的数量
parameters = initialize_parameters(layers_dims)
for i in range(0, num_iterations):
    # 前向传播
    a, caches = forward_propagation(X, parameters)
    # 计算代价
    cost_total = compute_cost(a, Y)  # m 的代价训练样本
    # 反向传播
    grads = backward_propagation(a, caches, parameters)
    # 更新参数
    parameters = update_parameters(parameters, grads)
    # 计算平均代价
    cost_avg = cost_total / m
        
```

- **随机梯度下降**：

```python
X = data_input
Y = labels
m = X.shape[1]  # 训练样本的数量
parameters = initialize_parameters(layers_dims)
for i in range(0, num_iterations):
    cost_total = 0
    for j in range(0, m):
        # 前向传播
        a, caches = forward_propagation(X[:,j], parameters)
        # 计算代价
        cost_total += compute_cost(a, Y[:,j])  # 一台训练样本的代价
        # 反向传播
        grads = backward_propagation(a, caches, parameters)
        # 更新参数
        parameters = update_parameters(parameters, grads)
    # 计算平均代价
    cost_avg = cost_total / m
```


在随机指标梯度下降中，你在更新梯度之前仅使用 1 训练样本。当训练集较大时，SGD可以更快。但参数会向最小值“振荡”，而不是平滑收敛。看起来是这样的： 

<img src="images/kiank_sgd.png" style="width:750px;height:250px;">
<div style="text-align:center;color:purple;"><u>  <b>图 1</b> </u> : <b>SGD 与 GD</b><br> “+”表示代价最小值。 SGD 会导致多次振荡才能达到收敛，但 SGD 的每一步计算速度都比 GD 快得多，因为它仅使用一个训练样本（相对于 GD 的整个批次）。</div>

**另请注意**，实现 SGD 总共需要 3 个 for 循环：
1. 超过迭代次数
2. $m$ 训练样本
3. 在各层上（更新所有参数，从 $(W^{[1]},b^{[1]})$ 到 $(W^{[L]},b^{[L]})$）

实际训练中，每次更新使用一部分样本，通常比使用整个训练集或单个样本更高效。mini-batch 梯度下降每次使用一小批样本，因此训练循环遍历的是各个 mini-batch，而非单独的训练样本。

<img src="images/kiank_minibatch.png" style="width:750px;height:250px;">
<div style="text-align:center;color:purple;"><u>  <b>图2</b> </u>： <b>SGD vs mini-batch GD</b><br> “+”表示代价最小值。在优化算法中使用 mini-batch 通常会带来更快的优化。</div>

<a name='3'></a>
## 3 - mini-batch 梯度下降

现在你将从训练集 (X, Y) 构建一些 mini-batch。

有两个步骤：
- **随机播放**：创建训练集 (X, Y) 的随机版本，如下所示。 X和Y的每一列代表一个训练样本。请注意，随机洗牌是在 X 和 Y 之间同步完成的。这样，洗牌后 X 的 $i^{th}$ 列就是对应于 Y 中的 $i^{th}$ 标签的示例。洗牌步骤确保示例将被随机拆分为不同的 mini-batch。 

<img src="images/kiank_shuffle.png" style="width:550px;height:300px;">

- **分区**：将打乱后的（X，Y）分区为大小为`mini_batch_size`（此处为64）的mini-batch。请注意，训练样本的数量并不总是能被 `mini_batch_size` 整除。最后的小批量可能会较小，但你无需担心这一点。当最终的 mini-batch 小于完整的 `mini_batch_size` 时，它将如下所示： 

<img src="images/kiank_partition.png" style="width:550px;height:300px;">

<a name='ex-2'></a>
### 练习 2 - random_mini_batches

实现`random_mini_batches`。洗牌部分已经为你编码了！为了帮助完成分区步骤，我们向你提供了以下代码，用于选择 $1^{st}$ 和 $2^{nd}$ mini-batch 的索引：
```python
first_mini_batch_X = shuffled_X[:, 0 : mini_batch_size]
second_mini_batch_X = shuffled_X[:, mini_batch_size : 2 * mini_batch_size]
...
```

请注意，最后一个 mini-batch 最终可能会小于 `mini_batch_size=64`。让 $\lfloor s \rfloor$ 表示 $s$ 向下舍入到最接近的整数（这是 Python 中的 `math.floor(s)`）。如果示例总数不是 `mini_batch_size=64` 的倍数，则将有 $\left\lfloor \frac{m}{mini\_batch\_size}\right\rfloor$ mini-batch，总共有 64 个示例，最终 mini-batch 中的示例数量将为 $\left(m-mini_\_batch_\_size \times \left\lfloor \frac{m}{mini\_batch\_size}\right\rfloor\right)$。 

**提示：**

$$
mini\_batch\_X = shuffled\_X[:, i : j]
$$

想出一种方法，可以使用 for 循环变量 `k` 帮助你以 mini_batch_size 的倍数递增 `i` 和 `j`。

例如，如果你想以 3 的倍数递增，你可以执行以下操作：

```python
n = 3
for k in (0 , 5):
    print(k * n)
```

In [ ]:
# GRADED FUNCTION: random_mini_batches

def random_mini_batches(X, Y, mini_batch_size = 64, seed = 0):
    """
    从 (X, Y) 创建随机 mini-batch 的列表
    
    参数：
    X -- 输入数据，形状（输入大小，示例数量）
    Y -- 真实的 "label" 向量（1 表示蓝点/0 表示红点），形状（1，示例数量）
    mini_batch_size -- mini-batch 的大小，整数
    
    返回：
    mini_batches -- 同步(mini_batch_X, mini_batch_Y)列表
    """
    
    np.random.seed(seed)            # 为了让你的"random" mini-batch 与我们的一样
    m = X.shape[1]                  # 训练样本的编号
    mini_batches = []
        
    # 步骤1：随机播放(X, Y)
    permutation = list(np.random.permutation(m))
    shuffled_X = X[:, permutation]
    shuffled_Y = Y[:, permutation].reshape((1, m))
    
    inc = mini_batch_size

    # 步骤 2 - 分区 (shuffled_X, shuffled_Y)。
    # 完整小批量大小的案例仅 i.e，每个案例 64 个。
    num_complete_minibatches = math.floor(m / mini_batch_size) # 分区中大小为 mini_batch_size 的小批量数量
    for k in range(0, num_complete_minibatches):
        # （约2行）
        # mini_batch_X =  
        # mini_batch_Y =
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        mini_batch = (mini_batch_X, mini_batch_Y)
        mini_batches.append(mini_batch)
    
    # 用于处理最终情况（最后 mini-batch < mini_batch_size i.e 小于 64）
    if m % mini_batch_size != 0:
        #（约2行）
        # mini_batch_X =
        # mini_batch_Y =
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        mini_batch = (mini_batch_X, mini_batch_Y)
        mini_batches.append(mini_batch)
    
    return mini_batches

In [ ]:
np.random.seed(1)
mini_batch_size = 64
nx = 12288
m = 148
X = np.array([x for x in range(nx * m)]).reshape((m, nx)).T
Y = np.random.randn(1, m) < 0.5

mini_batches = random_mini_batches(X, Y, mini_batch_size)
n_batches = len(mini_batches)

assert n_batches == math.ceil(m / mini_batch_size), f"Wrong number of mini batches. {n_batches} != {math.ceil(m / mini_batch_size)}"
for k in range(n_batches - 1):
    assert mini_batches[k][0].shape == (nx, mini_batch_size), f"Wrong shape in {k} mini batch for X"
    assert mini_batches[k][1].shape == (1, mini_batch_size), f"Wrong shape in {k} mini batch for Y"
    assert np.sum(np.sum(mini_batches[k][0] - mini_batches[k][0][0], axis=0)) == ((nx * (nx - 1) / 2 ) * mini_batch_size), "Wrong values. It happens if the order of X rows(features) changes"
if ( m % mini_batch_size > 0):
    assert mini_batches[n_batches - 1][0].shape == (nx, m % mini_batch_size), f"Wrong shape in the last minibatch. {mini_batches[n_batches - 1][0].shape} != {(nx, m % mini_batch_size)}"

assert np.allclose(mini_batches[0][0][0][0:3], [294912,  86016, 454656]), "Wrong values. Check the indexes used to form the mini batches"
assert np.allclose(mini_batches[-1][0][-1][0:3], [1425407, 1769471, 897023]), "Wrong values. Check the indexes used to form the mini batches"

print("\033[92mAll tests passed!")

In [ ]:
t_X, t_Y, mini_batch_size = random_mini_batches_test_case()
mini_batches = random_mini_batches(t_X, t_Y, mini_batch_size)

print ("shape of the 1st mini_batch_X: " + str(mini_batches[0][0].shape))
print ("shape of the 2nd mini_batch_X: " + str(mini_batches[1][0].shape))
print ("shape of the 3rd mini_batch_X: " + str(mini_batches[2][0].shape))
print ("shape of the 1st mini_batch_Y: " + str(mini_batches[0][1].shape))
print ("shape of the 2nd mini_batch_Y: " + str(mini_batches[1][1].shape)) 
print ("shape of the 3rd mini_batch_Y: " + str(mini_batches[2][1].shape))
print ("mini batch sanity check: " + str(mini_batches[0][0][0][0:3]))

random_mini_batches_test(random_mini_batches)

<div style="color:blue;">

**你应该记住什么**：
- 混洗和分区是构建 mini-batch 所需的两个步骤
- 2的幂通常被选择为mini-batch大小，例如 16, 32, 64, 128。

</div>

<a name='4'></a>
## 4 - Momentum（动量）

因为 mini-batch 梯度下降在看到示例的子集后进行参数更新，所以更新的方向存在一些方差，因此 mini-batch 梯度下降所采取的路径将“振荡”收敛。利用动量可以减少这些振荡。 

Momentum 考虑了过去的梯度来平滑更新。前面的梯度的“方向”存储在变量$v$中。正式而言，这将是梯度在之前步骤中的指数加权平均值。你还可以将 $v$ 视为球滚下山的“速度”，根据梯度/山坡的方向增加速度（和动量）。 

<img src="images/opt_momentum.png" style="width:400px;height:250px;">
<div style="text-align:center;color:purple;">

<u><b>图 3</b> </u>：红色箭头表示带动量的 mini-batch 梯度下降的更新方向，蓝点表示每一步在当前 mini-batch 上计算的梯度方向。更新时并非只依赖当前梯度，而是先用梯度更新动量 $v$，再根据 $v$ 更新参数。<br>

</div>


<a name='ex-3'></a>    
### 练习 3 - initialize_velocity
初始化速度。速度 $v$ 是一个 Python 字典，需要用零数组进行初始化。它的键与`grads`字典中的键相同，即：
对于 $l =1,...,L$：
```python
v["dW" + str(l)] = ... #（NumPy 与参数形状相同的零数组["W" + str(l)]）
v["db" + str(l)] = ... #（NumPy 与参数形状相同的零数组["b" + str(l)]）
```
**注意**，迭代器 l 在 for 循环中从 1 开始，因为第一个参数是 v["dW1"] 和 v["db1"] （即上标上的“1”）。

In [ ]:
# GRADED FUNCTION: initialize_velocity

def initialize_velocity(parameters):
    """
    将速度初始化为Python字典：
                - 键："dW1"、"db1"、...、"dWL"、"dbL" 
                - 值：NumPy 与相应的梯度/参数形状相同的零数组。
    参数：
    parameters -- 包含参数的Python字典。
                    parameters['W' + str(l)] = Wl
                    parameters['b' + str(l)] = bl
    
    返回：
    v -- 包含当前速度的Python字典。
                    v['dW' + str(l)] = dWl 的动量
                    v['db' + str(l)] = dbl 的动量
    """
    
    L = len(parameters) // 2 # 神经网络的层数
    v = {}
    
    # 初始化速度
    for l in range(1, L + 1):
        # （约2行）
        # v["dW" + str(l)] =
        # v["db" + str(l)] =
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
    return v

In [ ]:
parameters = initialize_velocity_test_case()

v = initialize_velocity(parameters)
print("v[\"dW1\"] =\n" + str(v["dW1"]))
print("v[\"db1\"] =\n" + str(v["db1"]))
print("v[\"dW2\"] =\n" + str(v["dW2"]))
print("v[\"db2\"] =\n" + str(v["db2"]))

initialize_velocity_test(initialize_velocity)

<a name='ex-4'></a>   
### 练习 4 - update_parameters_with_momentum

现在，用动量实现参数更新。对于$l = 1, ..., L$，动量更新规则为： 

$$ \begin{cases}
v_{dW^{[l]}} = \beta v_{dW^{[l]}} + (1 - \beta) dW^{[l]} \\
W^{[l]} = W^{[l]} - \alpha v_{dW^{[l]}}
\end{cases}\tag{3}$$

$$\begin{cases}
v_{db^{[l]}} = \beta v_{db^{[l]}} + (1 - \beta) db^{[l]} \\
b^{[l]} = b^{[l]} - \alpha v_{db^{[l]}} 
\end{cases}\tag{4}$$

其中L是层数，$\beta$是动量，$\alpha$是学习率。所有参数应存储在 `parameters` 字典中。  请注意，迭代器 `l` 在 `for` 循环中从 1 开始，因为第一个参数是 $W^{[1]}$ 和 $b^{[1]}$（即上标上的“1”）。

In [ ]:
# GRADED FUNCTION: update_parameters_with_momentum

def update_parameters_with_momentum(parameters, grads, v, beta, learning_rate):
    """
    使用 Momentum 更新参数
    
    参数：
    parameters -- 包含参数的Python字典：
                    parameters['W' + str(l)] = Wl
                    parameters['b' + str(l)] = bl
    grads -- 包含每个参数的梯度的 python 字典：
                    grads['dW' + str(l)] = dWl
                    grads['db' + str(l)] = dbl
    v -- 包含当前速度的 python 字典：
                    v['dW' + str(l)] = ...
                    v['db' + str(l)] = ...
    beta -- 动量超参数，标量
    learning_rate -- 学习率，标量
    
    返回：
    parameters -- 包含更新参数的Python字典 
    v -- 包含更新后的速度的 python 字典
    """

    L = len(parameters) // 2 # 神经网络的层数
    
    # 每个参数的动量更新
    for l in range(1, L + 1):
        
        # （约4行）
        # 计算速度
        # v["dW" + str(l)] = ...
        # v["db" + str(l)] = ...
        # 更新参数
        # parameters["W" + str(l)] = ...
        # parameters["b" + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
    return parameters, v

In [ ]:
parameters, grads, v = update_parameters_with_momentum_test_case()

parameters, v = update_parameters_with_momentum(parameters, grads, v, beta = 0.9, learning_rate = 0.01)
print("W1 = \n" + str(parameters["W1"]))
print("b1 = \n" + str(parameters["b1"]))
print("W2 = \n" + str(parameters["W2"]))
print("b2 = \n" + str(parameters["b2"]))
print("v[\"dW1\"] = \n" + str(v["dW1"]))
print("v[\"db1\"] = \n" + str(v["db1"]))
print("v[\"dW2\"] = \n" + str(v["dW2"]))
print("v[\"db2\"] = v" + str(v["db2"]))

update_parameters_with_momentum_test(update_parameters_with_momentum)

**注意**：
- 速度初始化为零。因此，该算法将需要几次迭代来“建立”速度并开始采取更大的步骤。
- 如果是$\beta = 0$，那么这就变成了标准的梯度下降，没有动量。 

**如何选择$\beta$？**

- $\beta$ 的动量越大，更新越顺利，因为它更多地考虑了过去的梯度。但如果 $\beta$ 太大，它也可能使更新过于平滑。 
- $\beta$ 的常用值范围为 0.8 到 0.999。如果你不想对此进行调整，$\beta = 0.9$ 通常是合理的默认值。 
- 为你的模型调整最佳 $\beta$ 可能需要尝试多个值，看看什么最适合降低代价函数 $J$ 的值。 

<div style="color:blue;">

**你应该记住什么**：
- 动量考虑了过去的梯度，以平滑梯度下降的步骤。它可以与批量梯度下降、mini-batch 梯度下降或随机梯度下降一起应用。
- 你必须调整动量超参数 $\beta$ 和学习率 $\alpha$。

</div>

<a name='5'></a>   
## 5 - Adam

Adam 是训练神经网络最有效的优化算法之一。它结合了 RMSProp（在讲座中描述）和 Momentum 的想法。 

**Adam 是如何工作的？**
1. 计算过去梯度的指数加权平均值，并将其存储在变量$v$（偏差修正前）和$v^{corrected}$（偏差修正后）中。 
2. 它计算过去梯度的平方的指数加权平均值，并将其存储在变量$s$（偏差修正前）和$s^{corrected}$（偏差修正后）中。 
3. 它根据“1”和“2”的信息组合在一个方向上更新参数。

对于$l = 1, ..., L$，更新规则为： 

$$
\begin{cases}
v_{dW^{[l]}} = \beta_1 v_{dW^{[l]}} + (1 - \beta_1) \frac{\partial \mathcal{J} }{ \partial W^{[l]} } \\
v^{corrected}_{dW^{[l]}} = \frac{v_{dW^{[l]}}}{1 - (\beta_1)^t} \\
s_{dW^{[l]}} = \beta_2 s_{dW^{[l]}} + (1 - \beta_2) (\frac{\partial \mathcal{J} }{\partial W^{[l]} })^2 \\
s^{corrected}_{dW^{[l]}} = \frac{s_{dW^{[l]}}}{1 - (\beta_2)^t} \\
W^{[l]} = W^{[l]} - \alpha \frac{v^{corrected}_{dW^{[l]}}}{\sqrt{s^{corrected}_{dW^{[l]}}} + \varepsilon}
\end{cases}
$$

其中：
- t 计算 Adam 走了的步数 
- L 是层数
- $\beta_1$ 和 $\beta_2$ 是控制两个指数加权平均值的超参数。 
- $\alpha$ 是学习率
- $\varepsilon$ 是一个非常小的数字，以避免被零除

和往常一样，所有参数都存储在 `parameters` 字典中  

<a name='ex-5'></a>   
### 练习 5 - initialize_adam

初始化 Adam 变量 $v, s$，用于跟踪过去的信息。

**说明**：变量 $v, s$ 是需要用零数组初始化的 python 字典。它们的密钥与 `grads` 相同，即：
对于 $l = 1, ..., L$：
```python
v["dW" + str(l)] = ... #（NumPy 与参数形状相同的零数组["W" + str(l)]）
v["db" + str(l)] = ... #（NumPy 与参数形状相同的零数组["b" + str(l)]）
s["dW" + str(l)] = ... #（NumPy 与参数形状相同的零数组["W" + str(l)]）
s["db" + str(l)] = ... #（NumPy 与参数形状相同的零数组["b" + str(l)]）

```

In [ ]:
# GRADED FUNCTION: initialize_adam

def initialize_adam(parameters) :
    """
    将 v 和 s 初始化为两个 python 字典：
                - 键："dW1"、"db1"、...、"dWL"、"dbL" 
                - 值：NumPy 与相应的梯度/参数形状相同的零数组。
    
    参数：
    parameters -- 包含参数的Python字典。
                    parameters["W" + str(l)] = Wl
                    parameters["b" + str(l)] = bl
    
    返回： 
    v -- python 字典，将包含梯度的指数加权平均值。用零初始化。
                    v["dW" + str(l)] = ...
                    v["db" + str(l)] = ...
    s -- python 字典，将包含平方梯度的指数加权平均值。用零初始化。
                    s["dW" + str(l)] = ...
                    s["db" + str(l)] = ...

    """
    
    L = len(parameters) // 2 # 神经网络的层数
    v = {}
    s = {}
    
    # 初始化v，s。输入："parameters"。输出："v, s"。
    for l in range(1, L + 1):
    # （约4行）
        # v["dW" + str(l)] = ...
        # v["db" + str(l)] = ...
        # s["dW" + str(l)] = ...
        # s["db" + str(l)] = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return v, s

In [ ]:
parameters = initialize_adam_test_case()

v, s = initialize_adam(parameters)
print("v[\"dW1\"] = \n" + str(v["dW1"]))
print("v[\"db1\"] = \n" + str(v["db1"]))
print("v[\"dW2\"] = \n" + str(v["dW2"]))
print("v[\"db2\"] = \n" + str(v["db2"]))
print("s[\"dW1\"] = \n" + str(s["dW1"]))
print("s[\"db1\"] = \n" + str(s["db1"]))
print("s[\"dW2\"] = \n" + str(s["dW2"]))
print("s[\"db2\"] = \n" + str(s["db2"]))

initialize_adam_test(initialize_adam)

<a name='ex-6'></a>   
### 练习 6 - update_parameters_with_adam

现在，用 Adam 来实现参数更新。回想一下，对于 $l = 1, ..., L$，一般更新规则是： 

$$\begin{cases}
v_{dW^{[l]}} = \beta_1 v_{dW^{[l]}} + (1 - \beta_1) \frac{\partial \mathcal{J} }{ \partial W^{[l]} } \\
v^{corrected}_{dW^{[l]}} = \frac{v_{dW^{[l]}}}{1 - (\beta_1)^t} \\
s_{dW^{[l]}} = \beta_2 s_{dW^{[l]}} + (1 - \beta_2) (\frac{\partial \mathcal{J} }{\partial W^{[l]} })^2 \\
s^{corrected}_{dW^{[l]}} = \frac{s_{dW^{[l]}}}{1 - (\beta_2)^t} \\
W^{[l]} = W^{[l]} - \alpha \frac{v^{corrected}_{dW^{[l]}}}{\sqrt{s^{corrected}_{dW^{[l]}}} + \varepsilon}
\end{cases}$$


**注意**，迭代器 `l` 在 `for` 循环中从 1 开始，因为第一个参数是 $W^{[1]}$ 和 $b^{[1]}$。 

In [ ]:
# GRADED FUNCTION: update_parameters_with_adam

def update_parameters_with_adam(parameters, grads, v, s, t, learning_rate = 0.01,
                                beta1 = 0.9, beta2 = 0.999,  epsilon = 1e-8):
    """
    使用 Adam 更新参数
    
    参数：
    parameters -- 包含参数的Python字典：
                    parameters['W' + str(l)] = Wl
                    parameters['b' + str(l)] = bl
    grads -- 包含每个参数的梯度的 python 字典：
                    grads['dW' + str(l)] = dWl
                    grads['db' + str(l)] = dbl
    v -- Adam变量，第一个梯度的移动平均值，python字典
    s -- Adam 变量，梯度平方的移动平均值，python 字典
    t -- Adam 变量，计算已采取的步数
    learning_rate -- 学习率，标量。
    beta1 -- 一阶矩估计的指数衰减超参数 
    beta2 -- 用于二阶矩估计的指数衰减超参数 
    epsilon -- 超参数防止 Adam 更新中被零除

    返回：
    parameters -- 包含更新参数的Python字典 
    v -- Adam变量，第一个梯度的移动平均值，python字典
    s -- Adam 变量，梯度平方的移动平均值，python 字典
    """
    
    L = len(parameters) // 2                 # 神经网络的层数
    v_corrected = {}                         # 初始化一阶矩估计，Python字典
    s_corrected = {}                         # 初始化二阶矩估计，python 字典
    
    # 对所有参数执行 Adam 更新
    for l in range(1, L + 1):
        # 梯度的移动平均值。输入："v, grads, beta1"。输出："v"。
        # （约2行）
        # v["dW" + str(l)] = ...
        # v["db" + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

        # 计算偏差修正的一阶矩估计。输入："v, beta1, t"。输出："v_corrected"。
        # （约2行）
        # v_corrected["dW" + str(l)] = ...
        # v_corrected["db" + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

        # 梯度平方的移动平均值。输入："s, grads, beta2"。输出："s"。
        #（约2行）
        # s["dW" + str(l)] = ...
        # s["db" + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

        # 计算经偏差修正的二阶原点矩估计。输入："s, beta2, t"。输出："s_corrected"。
        # （约2行）
        # s_corrected["dW" + str(l)] = ...
        # s_corrected["db" + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

        # 更新参数。输入："parameters, learning_rate, v_corrected, s_corrected, epsilon"。输出："parameters"。
        # （约2行）
        # parameters["W" + str(l)] = ...
        # parameters["b" + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

    return parameters, v, s, v_corrected, s_corrected

In [ ]:
parametersi, grads, vi, si, t, learning_rate, beta1, beta2, epsilon = update_parameters_with_adam_test_case()

parameters, v, s, vc, sc  = update_parameters_with_adam(parametersi, grads, vi, si, t, learning_rate, beta1, beta2, epsilon)
print(f"W1 = \n{parameters['W1']}")
print(f"W2 = \n{parameters['W2']}")
print(f"b1 = \n{parameters['b1']}")
print(f"b2 = \n{parameters['b2']}")

update_parameters_with_adam_test(update_parameters_with_adam)


**预期值：**
    
```
W1 = 
[[ 1.63937725 -0.62327448 -0.54308727]
 [-1.0578897   0.85032154 -2.31657668]]
W2 = 
[[ 0.33400549 -0.23563857]
 [ 1.47715417 -2.04561842]
 [-0.33729882 -0.36908457]]
b1 = 
[[ 1.72995096]
 [-0.7762447 ]]
b2 = 
[[ 1.14852557]
 [-1.08492339]
 [-0.15740527]]
```

你现在拥有三种有效的优化算法（mini-batch、梯度下降、Momentum、Adam）。让我们用每个优化器实现一个模型并观察差异。

<a name='6'></a>  
## 6 - 使用不同优化算法的模型

下面，你将使用以下“moons”数据集来测试不同的优化方法。 （该数据集被命名为“月亮”，因为这两个类中的每一个类的数据看起来都有点像新月形的月亮。） 

In [ ]:
train_X, train_Y = load_dataset()

3层神经网络已经为你实现了！你将使用以下方法训练它： 
- mini-batch **梯度下降**：它会调用你的函数：
    - `update_parameters_with_gd()`
- mini-batch **动量**：它会调用你的函数：
    - `initialize_velocity()` 和 `update_parameters_with_momentum()`
- mini-batch **Adam**：它会调用你的函数：
    - `initialize_adam()` 和 `update_parameters_with_adam()`

In [ ]:
def model(X, Y, layers_dims, optimizer, learning_rate = 0.0007, mini_batch_size = 64, beta = 0.9,
          beta1 = 0.9, beta2 = 0.999,  epsilon = 1e-8, num_epochs = 5000, print_cost = True):
    """
    3层神经网络模型，可以在不同的优化器模式下运行。
    
    参数：
    X -- 输入数据，形状（2，示例数量）
    Y -- 真实的 "label" 向量（1 表示蓝点/0 表示红点），形状（1，示例数量）
    optimizer -- 要传递的优化器，梯度下降，momentum 或 adam
    layers_dims -- python列表，包含每层的大小
    learning_rate -- 学习率，标量。
    mini_batch_size -- 小批量的大小
    beta -- 动量超参数
    beta1 -- 过去梯度估计的指数衰减超参数 
    beta2 -- 过去平方梯度估计值的指数衰减超参数 
    epsilon -- 超参数防止 Adam 更新中被零除
    num_epochs -- epoch数
    print_cost -- 真实打印每1000个epoch的代价

    返回：
    parameters -- 包含更新参数的Python字典 
    """

    L = len(layers_dims)             # 神经网络的层数
    costs = []                       # 跟踪代价
    t = 0                            # 初始化 Adam 更新所需的计数器
    seed = 10                        # 用于评分目的，以便你的 "random" mini-batch 与我们的相同
    m = X.shape[1]                   # 训练样本的编号
    
    # 初始化参数
    parameters = initialize_parameters(layers_dims)

    # 初始化优化器
    if optimizer == "gd":
        pass # 梯度下降无需初始化
    elif optimizer == "momentum":
        v = initialize_velocity(parameters)
    elif optimizer == "adam":
        v, s = initialize_adam(parameters)
    
    # 优化循环
    for i in range(num_epochs):
        
        # 定义随机mini-batch。我们增加种子以在每个epoch之后以不同的方式重新排列数据集
        seed = seed + 1
        minibatches = random_mini_batches(X, Y, mini_batch_size, seed)
        cost_total = 0
        
        for minibatch in minibatches:

            # 选择mini-batch
            (minibatch_X, minibatch_Y) = minibatch

            # 前向传播
            a3, caches = forward_propagation(minibatch_X, parameters)

            # 计算代价并添加到总代价中
            cost_total += compute_cost(a3, minibatch_Y)

            # 反向传播
            grads = backward_propagation(minibatch_X, minibatch_Y, caches)

            # 更新参数
            if optimizer == "gd":
                parameters = update_parameters_with_gd(parameters, grads, learning_rate)
            elif optimizer == "momentum":
                parameters, v = update_parameters_with_momentum(parameters, grads, v, beta, learning_rate)
            elif optimizer == "adam":
                t = t + 1 # Adam计数器
                parameters, v, s, _, _ = update_parameters_with_adam(parameters, grads, v, s,
                                                               t, learning_rate, beta1, beta2,  epsilon)
        cost_avg = cost_total / m
        
        # 打印每 1000 个 epoch 的代价
        if print_cost and i % 1000 == 0:
            print ("Cost after epoch %i: %f" %(i, cost_avg))
        if print_cost and i % 100 == 0:
            costs.append(cost_avg)
                
    # 绘制代价图
    plt.plot(costs)
    plt.ylabel('cost')
    plt.xlabel('epochs (per 100)')
    plt.title("Learning rate = " + str(learning_rate))
    plt.show()

    return parameters

现在，使用 3 种优化方法运行这个 3 层神经网络。

<a name='6-1'></a>  
### 6.1 - mini-batch 梯度下降

运行以下代码来查看模型如何处理 mini-batch 梯度下降。

In [ ]:
# 训练3层模型
layers_dims = [train_X.shape[0], 5, 2, 1]
parameters = model(train_X, train_Y, layers_dims, optimizer = "gd")

# 预测
predictions = predict(train_X, train_Y, parameters)

# 绘制决策边界
plt.title("Model with Gradient Descent optimization")
axes = plt.gca()
axes.set_xlim([-1.5,2.5])
axes.set_ylim([-1,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

<a name='6-2'></a>  
### 6.2 - 带动量的 mini-batch 梯度下降

接下来，运行以下代码来查看模型如何处理动量。因为这个例子相对简单，所以使用动量的收益很小 - 但对于更复杂的问题，你可能会看到更大的收益。

In [ ]:
# 训练3层模型
layers_dims = [train_X.shape[0], 5, 2, 1]
parameters = model(train_X, train_Y, layers_dims, beta = 0.9, optimizer = "momentum")

# 预测
predictions = predict(train_X, train_Y, parameters)

# 绘制决策边界
plt.title("Model with Momentum optimization")
axes = plt.gca()
axes.set_xlim([-1.5,2.5])
axes.set_ylim([-1,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

<a name='6-3'></a>  
### 6.3 - 使用 Adam 的 mini-batch 优化

最后，运行以下代码来查看模型与 Adam 的配合情况。

In [ ]:
# 训练3层模型
layers_dims = [train_X.shape[0], 5, 2, 1]
parameters = model(train_X, train_Y, layers_dims, optimizer = "adam")

# 预测
predictions = predict(train_X, train_Y, parameters)

# 绘制决策边界
plt.title("Model with Adam optimization")
axes = plt.gca()
axes.set_xlim([-1.5,2.5])
axes.set_ylim([-1,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

<a name='6-4'></a>  
### 6.4 - 总结

<table> 
    <tr>
        <td>
        <b>优化方法</b>
        </td>
        <td>
        <b>精度</b>
        </td>
        <td>
        <b>代价形状</b>
        </td>
    </tr>
        <td>
        梯度下降
        </td>
        <td>
        >71%
        </td>
        <td>
        光滑
        </td>
    <tr>
        <td>
        Momentum（动量）
        </td>
        <td>
        >71%
        </td>
        <td>
        光滑
        </td>
    </tr>
    <tr>
        <td>
        Adam
        </td>
        <td>
        >94%
        </td>
        <td>
        更顺畅
        </td>
    </tr>
</table> 

动量通常会有所帮助，但考虑到学习率小和简单的数据集，其影响几乎可以忽略不计。

另一方面，Adam 明显优于 mini-batch 梯度下降和 Momentum。如果你在这个简单的数据集上运行模型更多的epoch，所有三种方法都将产生非常好的结果。然而，你已经看到 Adam 的收敛速度要快得多。

Adam 的一些优点包括：

- 内存要求相对较低（虽然比梯度下降和梯度下降有动量） 
- 即使很少调整超参数，通常也能很好地工作（$\alpha$ 除外）

**参考文献**：

- Adam纸：https://arxiv.org/pdf/1412.6980.pdf

<a name='7'></a>  
## 7 - 学习率衰减和调度

最后，学习率是另一个可以帮助你加快学习速度的超参数。 

在训练的第一部分中，你的模型可以采取大步长，但随着时间的推移，使用学习率 alpha 的固定值可能会导致你的模型陷入宽振荡，永远不会完全收敛。但是，如果你要随着时间的推移慢慢减少学习率 alpha，那么你可以采取更小、更慢的步骤，使你更接近最小值。这就是学习率衰减背后的想法。 

学习率衰减可以通过使用自适应方法或预定义的学习率时间表来实现。 

现在，你将在三种不同的优化器模式下将计划的学习率衰减应用于 3 层神经网络，并查看每种模式有何不同，以及在不同epoch的调度效果。 

该模型本质上与你之前使用的模型相同，只是在此模型中你将能够包含学习率衰减。它包括两个新参数：衰减和 decay_rate。 

In [ ]:
def model(X, Y, layers_dims, optimizer, learning_rate = 0.0007, mini_batch_size = 64, beta = 0.9,
          beta1 = 0.9, beta2 = 0.999,  epsilon = 1e-8, num_epochs = 5000, print_cost = True, decay=None, decay_rate=1):
    """
    3层神经网络模型，可以在不同的优化器模式下运行。
    
    参数：
    X -- 输入数据，形状（2，示例数量）
    Y -- 真实的 "label" 向量（1 表示蓝点/0 表示红点），形状（1，示例数量）
    layers_dims -- python列表，包含每层的大小
    learning_rate -- 学习率，标量。
    mini_batch_size -- 小批量的大小
    beta -- 动量超参数
    beta1 -- 过去梯度估计的指数衰减超参数 
    beta2 -- 过去平方梯度估计值的指数衰减超参数 
    epsilon -- 超参数防止 Adam 更新中被零除
    num_epochs -- epoch数
    print_cost -- 真实打印每1000个epoch的代价

    返回：
    parameters -- 包含更新参数的Python字典 
    """

    L = len(layers_dims)             # 神经网络的层数
    costs = []                       # 跟踪代价
    t = 0                            # 初始化 Adam 更新所需的计数器
    seed = 10                        # 用于评分目的，以便你的 "random" mini-batch 与我们的相同
    m = X.shape[1]                   # 训练样本的编号
    lr_rates = []
    learning_rate0 = learning_rate   # 原版学习率
    
    # 初始化参数
    parameters = initialize_parameters(layers_dims)

    # 初始化优化器
    if optimizer == "gd":
        pass # 梯度下降无需初始化
    elif optimizer == "momentum":
        v = initialize_velocity(parameters)
    elif optimizer == "adam":
        v, s = initialize_adam(parameters)
    
    # 优化循环
    for i in range(num_epochs):
        
        # 定义随机mini-batch。我们增加种子以在每个epoch之后以不同的方式重新排列数据集
        seed = seed + 1
        minibatches = random_mini_batches(X, Y, mini_batch_size, seed)
        cost_total = 0
        
        for minibatch in minibatches:

            # 选择mini-batch
            (minibatch_X, minibatch_Y) = minibatch

            # 前向传播
            a3, caches = forward_propagation(minibatch_X, parameters)

            # 计算代价并添加到总代价中
            cost_total += compute_cost(a3, minibatch_Y)

            # 反向传播
            grads = backward_propagation(minibatch_X, minibatch_Y, caches)

            # 更新参数
            if optimizer == "gd":
                parameters = update_parameters_with_gd(parameters, grads, learning_rate)
            elif optimizer == "momentum":
                parameters, v = update_parameters_with_momentum(parameters, grads, v, beta, learning_rate)
            elif optimizer == "adam":
                t = t + 1 # Adam计数器
                parameters, v, s, _, _ = update_parameters_with_adam(parameters, grads, v, s,
                                                               t, learning_rate, beta1, beta2,  epsilon)
        cost_avg = cost_total / m
        if decay:
            learning_rate = decay(learning_rate0, i, decay_rate)
        # 打印每 1000 个 epoch 的代价
        if print_cost and i % 1000 == 0:
            print ("Cost after epoch %i: %f" %(i, cost_avg))
            if decay:
                print("learning rate after epoch %i: %f"%(i, learning_rate))
        if print_cost and i % 100 == 0:
            costs.append(cost_avg)
                
    # 绘制代价图
    plt.plot(costs)
    plt.ylabel('cost')
    plt.xlabel('epochs (per 100)')
    plt.title("Learning rate = " + str(learning_rate))
    plt.show()

    return parameters

<a name='7-1'></a>  
### 7.1 - 每次迭代进行学习率衰减  

对于作业的这一部分，你将尝试学习率衰减的预定义计划之一，称为指数学习率衰减。它采用以下数学形式：

$$\alpha = \frac{1}{1 + decayRate \times epochNumber} \alpha_{0}$$

<a name='ex-7'></a>  
### 练习 7 - update_lr

使用指数权重衰减计算新的学习率。

In [ ]:
# GRADED FUNCTION: update_lr

def update_lr(learning_rate0, epoch_num, decay_rate):
    """
    使用指数权重衰减计算更新的学习率。
    
    参数：
    learning_rate0 -- 原装学习率。标量
    epoch_num -- epoch 数。整数
    decay_rate -- 衰减率。标量

    返回：
    learning_rate -- 更新了学习率。标量 
    """
    #（约1行）
    # learning_rate = 
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return learning_rate

In [ ]:
learning_rate = 0.5
print("Original learning rate: ", learning_rate)
epoch_num = 2
decay_rate = 1
learning_rate_2 = update_lr(learning_rate, epoch_num, decay_rate)

print("Updated learning rate: ", learning_rate_2)

update_lr_test(update_lr)

In [ ]:
# 训练3层模型
layers_dims = [train_X.shape[0], 5, 2, 1]
parameters = model(train_X, train_Y, layers_dims, optimizer = "gd", learning_rate = 0.1, num_epochs=5000, decay=update_lr)

# 预测
predictions = predict(train_X, train_Y, parameters)

# 绘制决策边界
plt.title("Model with Gradient Descent optimization")
axes = plt.gca()
axes.set_xlim([-1.5,2.5])
axes.set_ylim([-1,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

请注意，如果你将衰减设置为在每次迭代时发生，则学习率会很快变为零 - 即使你从较高的学习率开始也是如此。 
<table> 
    <tr>
        <td>
        <b>Eepoch编号</b>
        </td>
        <td>
        <b>学习率</b>
        </td>
        <td>
        <b>代价</b>
        </td>
    </tr>
    <tr>
        <td>
        0
        </td>
        <td>
        0.100000
        </td>
        <td>
        0.701091
        </td>
    </tr>
    <tr>
        <td>
        1000
        </td>
        <td>
        0.000100
        </td>
        <td>
        0.661884
        </td>
    </tr>
    <tr>
        <td>
        2000
        </td>
        <td>
        0.000050
        </td>
        <td>
        0.658620
        </td>
    </tr>
    <tr>
        <td>
        3000
        </td>
        <td>
        0.000033
        </td>
        <td>
        0.656765
        </td>
    </tr>
    <tr>
        <td>
        4000
        </td>
        <td>
        0.000025
        </td>
        <td>
        0.655486
        </td>
    </tr>
    <tr>
        <td>
        5000
        </td>
        <td>
        0.000020
        </td>
        <td>
        0.654514
        </td>
    </tr>
</table> 

当你训练几个epoch时，这不会造成很多麻烦，但是当epoch数量很大时，优化算法将停止更新。解决此问题的一种常见方法是每隔几步衰减学习率。这称为固定间隔调度。

<a name='7-2'></a> 
### 7.2 - 固定间隔调度

你可以通过以固定时间间隔（例如 1000）安排指数学习率衰减来帮助防止学习率过快地加速到零。你可以对间隔进行编号，或者将历元除以时间间隔，即具有常量学习率的窗口大小。 

<img src="images/lr.png" style="width:400px;height:250px;">

<a name='ex-8'></a> 
### 练习 8 - schedule_lr_decay

使用指数权重衰减和固定间隔调度来计算新的学习率。

**说明**：实现学习率调度，使其仅在 epochNum 是 timeInterval 的倍数时发生变化。

**注：** 分母中的分数采用取整运算。 

$$\alpha = \frac{1}{1 + decayRate \times \lfloor\frac{epochNum}{timeInterval}\rfloor} \alpha_{0}$$

**提示：** [numpy.floor](https://numpy.org/doc/stable/reference/generated/numpy.floor.html)

In [ ]:
# GRADED FUNCTION: schedule_lr_decay

def schedule_lr_decay(learning_rate0, epoch_num, decay_rate, time_interval=1000):
    """
    使用指数权重衰减计算更新的学习率。
    
    参数：
    learning_rate0 -- 原装学习率。标量
    epoch_num -- epoch 数。整数。
    decay_rate -- 衰减率。标量。
    time_interval -- 更新学习率的epoch数。

    返回：
    learning_rate -- 更新了学习率。标量 
    """
    # （约1行）
    # learning_rate = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return learning_rate

In [ ]:
learning_rate = 0.5
print("Original learning rate: ", learning_rate)

epoch_num_1 = 10
epoch_num_2 = 100
decay_rate = 0.3
time_interval = 100
learning_rate_1 = schedule_lr_decay(learning_rate, epoch_num_1, decay_rate, time_interval)
learning_rate_2 = schedule_lr_decay(learning_rate, epoch_num_2, decay_rate, time_interval)
print("Updated learning rate after {} epochs: ".format(epoch_num_1), learning_rate_1)
print("Updated learning rate after {} epochs: ".format(epoch_num_2), learning_rate_2)

schedule_lr_decay_test(schedule_lr_decay)

**预期输出**
```
Original learning rate:  0.5
Updated learning rate after 10 epochs:  0.5
Updated learning rate after 100 epochs:  0.3846153846153846
```

<a name='7-3'></a> 
### 7.3 - 对每种优化方法使用学习率衰减

下面，你将使用以下“moons”数据集来测试不同的优化方法。 （该数据集被命名为“月亮”，因为这两个类中的每一个类的数据看起来都有点像新月形的月亮。） 

<a name='7-3-1'></a> 
#### 7.3.1 - 梯度下降与学习率衰减

运行以下代码来查看模型如何进行梯度下降和学习率衰减。

In [ ]:
# 训练3层模型
layers_dims = [train_X.shape[0], 5, 2, 1]
parameters = model(train_X, train_Y, layers_dims, optimizer = "gd", learning_rate = 0.1, num_epochs=5000, decay=schedule_lr_decay)

# 预测
predictions = predict(train_X, train_Y, parameters)

# 绘制决策边界
plt.title("Model with Gradient Descent optimization")
axes = plt.gca()
axes.set_xlim([-1.5,2.5])
axes.set_ylim([-1,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

<a name='7-3-2'></a> 
#### 7.3.2 - 带动量和学习率衰减的梯度下降

运行以下代码来查看模型如何进行梯度下降动量和学习率衰减。

In [ ]:
# 训练3层模型
layers_dims = [train_X.shape[0], 5, 2, 1]
parameters = model(train_X, train_Y, layers_dims, optimizer = "momentum", learning_rate = 0.1, num_epochs=5000, decay=schedule_lr_decay)

# 预测
predictions = predict(train_X, train_Y, parameters)

# 绘制决策边界
plt.title("Model with Gradient Descent with momentum optimization")
axes = plt.gca()
axes.set_xlim([-1.5,2.5])
axes.set_ylim([-1,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

<a name='7-3-3'></a> 
#### 7.3.3 - 带学习率衰减的 Adam

运行以下代码来查看模型如何进行 Adam 和学习率衰减。

In [ ]:
# 训练3层模型
layers_dims = [train_X.shape[0], 5, 2, 1]
parameters = model(train_X, train_Y, layers_dims, optimizer = "adam", learning_rate = 0.01, num_epochs=5000, decay=schedule_lr_decay)

# 预测
predictions = predict(train_X, train_Y, parameters)

# 绘制决策边界
plt.title("Model with Adam optimization")
axes = plt.gca()
axes.set_xlim([-1.5,2.5])
axes.set_ylim([-1,1.5])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

<a name='7-4'></a> 
### 7.4 - 用不同的方法实现相似的性能

使用 mini-batch GD 或带有 Momentum 的 mini-batch GD，精度明显低于 Adam，但是当在上面添加学习率衰减时，两者都可以实现与 Adam 类似的速度和精度分数。

在 Adam 的例子中，请注意学习曲线达到了类似的准确性，但速度更快。

<table> 
    <tr>
        <td>
        <b>优化方法</b>
        </td>
        <td>
        <b>精度</b>
        </td>
    </tr>
        <td>
        梯度下降
        </td>
        <td>
        >94.6%
        </td>
    <tr>
        <td>
        Momentum（动量）
        </td>
        <td>
        >95.6%
        </td>
    </tr>
    <tr>
        <td>
        Adam
        </td>
        <td>
        94%
        </td>
    </tr>
</table> 

**恭喜**！你已完成优化方法 Notebook 的结尾。以下是你现在可以执行的所有操作的快速回顾： 

* 对模型应用三种不同的优化方法 
* 为你的训练集构建 mini-batch 
* 使用学习率衰减调度来加快训练速度

干得好！